# Лабораторна №3

## 1. Встановлення залежностей

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q transformers datasets pytorch-lightning jiwer gdown
!pip install -q accelerate soundfile librosa evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


## Імпорти

In [ ]:
import os
import json
import zipfile
from dataclasses import dataclass
from typing import Any, Dict, List, Union

import gdown
import torch
import jiwer
import numpy as np
import pytorch_lightning as pl

from datasets import load_dataset, Audio, DatasetDict
from torch.utils.data import DataLoader
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    get_linear_schedule_with_warmup,
)
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import CSVLogger

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


## Завантаження датасету Toronto

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/dataset2.zip'

if not os.path.exists('toronto_data'):
    print("Розпаковуємо датасет...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('toronto_data')
    print("Датасет розпаковано!")
else:
    print("Датасет вже існує.")

Розпаковуємо датасет...
Датасет розпаковано!


In [ ]:
file_id = '1BmFqkPABsodsAyvAFLF8V4PyVPbzXMhd'
url = f'https://drive.google.com/uc?/id={file_id}'
output = 'toronto_dataset.zip'

if not os.path.exists(output):
    gdown.download(url, output, quiet=False)

if not os.path.exists('toronto_data'):
    with zipfile.ZipFile(output, 'r') as zip_ref:
        zip_ref.extractall('toronto_data')
else:
    print("Датасет вже існує, пропускаємо розпакування")

Downloading...
From: https://drive.google.com/uc?/id=1BmFqkPABsodsAyvAFLF8V4PyVPbzXMhd
To: /content/toronto_dataset.zip
1.70kB [00:00, 3.54MB/s]

Датасет вже існує, пропускаємо розпакування


## Підготовка датасету

In [ ]:
MODEL_NAME = "openai/whisper-small"  
LANGUAGE = "uk"                     
TASK = "transcribe"
SAMPLING_RATE = 16000
BATCH_SIZE = 8
LEARNING_RATE = 1e-5
MAX_EPOCHS = 3

TEST_IDS = [
    'toronto_27', 'toronto_46', 'toronto_42', 'toronto_37', 'toronto_89',
    'toronto_43', 'toronto_157', 'toronto_9', 'toronto_156', 'toronto_7',
    'toronto_123', 'toronto_54', 'toronto_67', 'toronto_62', 'toronto_81',
    'toronto_134', 'toronto_148', 'toronto_21', 'toronto_135', 'toronto_166',
    'toronto_58'
]

In [ ]:
transcripts = {}
with open("toronto_data/labels.jsonl", "r", encoding="utf-8") as f:
    data = json.loads(f.readline())
    for path, text in data.items():
        file_name = os.path.basename(path)
        transcripts[file_name] = text

print(f"Всього транскрипцій: {len(transcripts)}")
for i, (k, v) in enumerate(list(transcripts.items())[:3]):
    print(f"  {k}: {v[:80]}...")

Всього транскрипцій: 29232
  toronto_157_0.wav: Слава Ісу! Ви сі дивите програму «Грати, песик, дужка, гривня, знак питання, дол...
  toronto_157_1.wav: Купол Верховної Ради впав під час засідання, накривши сотні депутатів. Всередині...
  toronto_157_2.wav: живих добивають з автоматів, кров, вогонь, агонія... і Надія. Народний депутат Н...


In [ ]:
def get_sample_id(path):
    """Витягує ID зразка типу 'toronto_27' з шляху до файлу"""
    fname = os.path.basename(path)
    return "_".join(fname.split("_")[:2])

raw_dataset = load_dataset("audiofolder", data_dir="toronto_data")
raw_dataset = raw_dataset.cast_column("audio", Audio(decode=False))

def is_test(example):
    return get_sample_id(example["audio"]["path"]) in TEST_IDS

test_ds     = raw_dataset['train'].filter(is_test)
train_val_ds = raw_dataset['train'].filter(lambda x: not is_test(x))

train_val_split = train_val_ds.train_test_split(test_size=0.1, seed=42)

dataset = DatasetDict({
    "train":      train_val_split["train"],
    "validation": train_val_split["test"],
    "test":       test_ds,
})

print(f"Train:      {len(dataset['train'])} зразків")
print(f"Validation: {len(dataset['validation'])} зразків")
print(f"Test:       {len(dataset['test'])} зразків")

Resolving data files:   0%|          | 0/18304 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Filter:   0%|          | 0/18303 [00:00<?, ? examples/s]

Train:      11484 зразків
Validation: 1277 зразків
Test:       5542 зразків


In [ ]:
def add_transcript(example):
    file_name = os.path.basename(example["audio"]["path"])
    example["sentence"] = transcripts.get(file_name, "")
    return example

dataset = dataset.map(add_transcript)
dataset = dataset.filter(lambda x: x["sentence"] != "")
dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLING_RATE, decode=True))

processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK
)

def prepare_dataset(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_features[0]
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch

encoded_dataset = dataset.map(
    prepare_dataset,
    remove_columns=["audio", "label", "sentence"],
    num_proc=2
)

Map:   0%|          | 0/11484 [00:00<?, ? examples/s]

Map:   0%|          | 0/1277 [00:00<?, ? examples/s]

Map:   0%|          | 0/5542 [00:00<?, ? examples/s]

Filter:   0%|          | 0/11484 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1277 [00:00<?, ? examples/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map (num_proc=2):   0%|          | 0/11417 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1272 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5521 [00:00<?, ? examples/s]

## DataCollator та DataLoaders

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1), -100
        )
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

train_loader = DataLoader(
    encoded_dataset["train"],
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)
val_loader = DataLoader(
    encoded_dataset["validation"],
    batch_size=BATCH_SIZE,
    collate_fn=data_collator,
    num_workers=2,
    pin_memory=True,
)
test_loader = DataLoader(
    encoded_dataset["test"],
    batch_size=4,             
    collate_fn=data_collator,
    num_workers=2,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 1428
Val batches:   159
Test batches:  1381


In [ ]:
import logging

## PyTorch Lightning Модель

In [ ]:
class WhisperLightningModule(pl.LightningModule):
    def __init__(
        self,
        model_name_or_path: str = "openai/whisper-small",
        learning_rate: float = 1e-5,
        warmup_steps: int = 100,
        total_steps: int = 1000,
    ):
        super().__init__()
        self.save_hyperparameters()

        self.model = WhisperForConditionalGeneration.from_pretrained(model_name_or_path)

        self.model.config.use_cache = False

        self.model.generation_config.language = LANGUAGE
        self.model.generation_config.task = TASK
        self.model.generation_config.forced_decoder_ids = None 
        self.model.generation_config.suppress_tokens = []

        self.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language=LANGUAGE, task=TASK
        )

        self.val_predictions = []
        self.val_references  = []
        self.test_predictions = []
        self.test_references  = []

    def forward(self, input_features, labels=None):
        return self.model(input_features=input_features, labels=labels)

    def training_step(self, batch, batch_idx):
        outputs = self(input_features=batch["input_features"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss, prog_bar=True, on_step=True, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        outputs = self(input_features=input_features, labels=labels)
        self.log("val_loss", outputs.loss, prog_bar=True, sync_dist=True)

        generated_ids = self.model.generate(
            input_features,
            forced_decoder_ids=self.forced_decoder_ids,
        )

        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = processor.tokenizer.pad_token_id

        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        refs  = processor.batch_decode(labels_for_decode, skip_special_tokens=True)

        self.val_predictions.extend(preds)
        self.val_references.extend(refs)


    def on_validation_epoch_end(self):
        if not self.val_predictions:
            return

        wer = jiwer.wer(self.val_references, self.val_predictions)
        cer = jiwer.cer(self.val_references, self.val_predictions)

        self.log("val_wer", wer, prog_bar=True, sync_dist=True)
        self.log("val_cer", cer, prog_bar=True, sync_dist=True)


        logging.getLogger("pytorch_lightning").info(f"Validation Metrics - WER: {wer:.4f}, CER: {cer:.4f}")

        self.val_predictions.clear()
        self.val_references.clear()

    def test_step(self, batch, batch_idx):
        input_features = batch["input_features"]
        labels = batch["labels"]

        generated_ids = self.model.generate(
            input_features,
            forced_decoder_ids=self.forced_decoder_ids,
        )

        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = processor.tokenizer.pad_token_id

        preds = processor.batch_decode(generated_ids, skip_special_tokens=True)
        refs  = processor.batch_decode(labels_for_decode, skip_special_tokens=True)

        self.test_predictions.extend(preds)
        self.test_references.extend(refs)

    def on_test_epoch_end(self):
        wer = jiwer.wer(self.test_references, self.test_predictions)
        cer = jiwer.cer(self.test_references, self.test_predictions)
        self.log("test_wer", wer)
        self.log("test_cer", cer)

        print("\n" + "="*60)
        print("           РЕЗУЛЬТАТИ НА ТЕСТОВІЙ ВИБІРЦІ")
        print("="*60)
        print(f"  WER (Word Error Rate):      {wer:.4f}  ({wer*100:.2f}%)")
        print(f"  CER (Character Error Rate): {cer:.4f}  ({cer*100:.2f}%)")
        print("="*60)

        print("\nПриклади предикцій:")
        for ref, pred in zip(self.test_references[:5], self.test_predictions[:5]):
            print(f"  REF:  {ref}")
            print(f"  PRED: {pred}")
            print()

        self.test_predictions.clear()
        self.test_references.clear()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.learning_rate,
            weight_decay=1e-2,
        )
        scheduler = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps=self.hparams.warmup_steps,
            num_training_steps=self.hparams.total_steps,
        )
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "interval": "step"},
        }

## Тренування

In [ ]:
total_steps = MAX_EPOCHS * len(train_loader)
warmup_steps = total_steps // 10FF
print(f"Total training steps: {total_steps}, Warmup: {warmup_steps}")

model = WhisperLightningModule(
    model_name_or_path=MODEL_NAME,
    learning_rate=LEARNING_RATE,
    warmup_steps=warmup_steps,
    total_steps=total_steps,
)


checkpoint_callback = ModelCheckpoint(
    dirpath="/content/drive/MyDrive/whisper_checkpoints",
    filename="whisper-best-{epoch:02d}-wer{val_wer:.3f}",
    save_top_k=1,
    monitor="val_wer",
    mode="min",
    verbose=True,
)

early_stopping = EarlyStopping(
    monitor="val_wer",
    patience=2,      
    mode="min",
    verbose=True,
)

logger = CSVLogger("logs", name="whisper_toronto")

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, early_stopping],
    logger=logger,
    log_every_n_steps=10,
    precision="16-mixed",
    gradient_clip_val=1.0,
    enable_progress_bar=False, 
)

print("Починаємо тренування...")
trainer.fit(model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Total training steps: 4284, Warmup: 428


model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

INFO:pytorch_lightning.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Починаємо тренування...


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/model_summary/model_summary.py:242: Precision 16-mixed is not supported by the model summary.  Estimated model size in MB will not be accurate. Using 32 bits instead.


┏━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━┳━━━━━━━┓
┃   ┃ Name  ┃ Type                            ┃ Params ┃ Mode ┃ FLOPs ┃
┡━━━╇━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━╇━━━━━━━┩
│ 0 │ model │ WhisperForConditionalGeneration │  241 M │ eval │     0 │
└───┴───────┴─────────────────────────────────┴────────┴──────┴───────┘

Trainable params: 241 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 241 M                                                                                                
Total estimated model params size (MB): 966                                                                        
Modules in train mode: 0                                                                                           
Modules in eval mode: 350                                                                                          
Total FLOPs: 0

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAt

## 8. Тестування найкращої моделі

In [ ]:
best_model_path = checkpoint_callback.best_model_path
print(f"Завантажуємо найкращу модель: {best_model_path}")

best_model = WhisperLightningModule.load_from_checkpoint(
    best_model_path,
    model_name_or_path=MODEL_NAME,
)

test_results = trainer.test(best_model, dataloaders=test_loader)

Завантажуємо найкращу модель: /content/drive/MyDrive/whisper_checkpoints/whisper-best-epoch=01-werval_wer=0.313.ckpt


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]



           РЕЗУЛЬТАТИ НА ТЕСТОВІЙ ВИБІРЦІ
  WER (Word Error Rate):      0.3808  (38.08%)
  CER (Character Error Rate): 0.2173  (21.73%)

Приклади предикцій:
  REF:  Слава Ісу! Ви дивитесь програму «#@)₴?$0». Я є її ведучий – Майкл Щур. Вйо до новин!
  PRED: Слава Ісу! Ви дивитесь програму «#@)₴?$0». Я є її ведучий – Майкл Щур. Вйо до новин!

  REF:  Валарморгулісне! Україна за три тижні до виборів президента та за п’ять тижнів до нового сезону «Гри престолів».
  PRED: «Фаларморгу́лісне»! Україна за три тижні до виборів президента та за п’ять тижнів до нового сезону «Гри престолів»!

  REF:  стан людини, яка на виборчій дільниці отримала бюлетень довжиною більше метра.
  PRED: Стан людини, яка на виборчій дільниці отримала бюлетень довжиною більше метра.

  REF:  Ну, а хто ж на верхівці цього чучундрологічного ланцюга? На жаль, про цю істоту нам нічого не відомо.
  PRED: Ну, а хто ж на верхівці цього чучиндрологічного ланцюга? На жаль, пресюністу додам нічого не відомо.

  REF:  Саме т

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_cer          │    0.21734115481376648    │
│         test_wer          │    0.3807894289493561     │
└───────────────────────────┴───────────────────────────┘

## 9. Baseline (без fine-tuning) для порівняння

In [ ]:
print("Оцінюємо базовий Whisper (без fine-tuning)...")

baseline_processor = WhisperProcessor.from_pretrained(
    MODEL_NAME, language=LANGUAGE, task=TASK
)
baseline_model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME)
baseline_model.eval()

forced_ids = baseline_processor.get_decoder_prompt_ids(language=LANGUAGE, task=TASK)

if torch.cuda.is_available():
    baseline_model = baseline_model.cuda()

baseline_preds, baseline_refs = [], []

with torch.no_grad():
    for batch in test_loader:
        input_features = batch["input_features"]
        labels = batch["labels"]

        if torch.cuda.is_available():
            input_features = input_features.cuda()

        gen_ids = baseline_model.generate(input_features, forced_decoder_ids=forced_ids)

        labels[labels == -100] = baseline_processor.tokenizer.pad_token_id
        preds = baseline_processor.batch_decode(gen_ids, skip_special_tokens=True)
        refs  = baseline_processor.batch_decode(labels, skip_special_tokens=True)

        baseline_preds.extend(preds)
        baseline_refs.extend(refs)

baseline_wer = jiwer.wer(baseline_refs, baseline_preds)
baseline_cer = jiwer.cer(baseline_refs, baseline_preds)

print("\n" + "="*60)
print("           ПОРІВНЯННЯ МОДЕЛЕЙ")
print("="*60)
print(f"  {'Модель':<35} {'WER':>8} {'CER':>8}")
print("-"*60)
print(f"  {'Whisper-small (pretrained, без FT)':<35} {baseline_wer*100:>7.2f}% {baseline_cer*100:>7.2f}%")
finetuned_wer = test_results[0].get('test_wer', 0)
finetuned_cer = test_results[0].get('test_cer', 0)
print(f"  {'Whisper-small (fine-tuned, наш)':<35} {finetuned_wer*100:>7.2f}% {finetuned_cer*100:>7.2f}%")
print("-"*60)
print(f"  Покращення WER: {(baseline_wer - finetuned_wer)*100:.2f} п.п.")
print(f"  Покращення CER: {(baseline_cer - finetuned_cer)*100:.2f} п.п.")
print("="*60)

Оцінюємо базовий Whisper (без fine-tuning)...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.



           ПОРІВНЯННЯ МОДЕЛЕЙ
  Модель                                   WER      CER
------------------------------------------------------------
  Whisper-small (pretrained, без FT)    45.71%   23.39%
  Whisper-small (fine-tuned, наш)       38.08%   21.73%
------------------------------------------------------------
  Покращення WER: 7.64 п.п.
  Покращення CER: 1.66 п.п.
